# Plot results for offline prompting in the "farm" example

In [1]:
%cd ..
%pwd  # should be "llm-adaptation"

C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation


C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'C:\\Users\\micha\\OneDrive - Univerzita Karlova\\research\\2024-LLM-DEECo\\llm-adaptation'

In [22]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import sys

In [23]:
results_folder = Path("generated_adaptations/farm")

## Run experiments

In [24]:
prompts = {
    "default": "generated_adaptations/prompts/farm.txt",
    "high": "generated_adaptations/prompts/farm_strategy.txt",  # high-level strategy
    "state": "generated_adaptations/prompts/farm_state.txt",  # example of environment state
    "less": "generated_adaptations/prompts/farm_less.txt",  # without access to the `arriving_drones` and `protecting_drones` attributes of Field
}
repeats = 2
start = 2

In [25]:
# import generated_adaptations.generator as generator

In [26]:
for variant, prompt in prompts.items():
    for repeat in range(start, start + repeats):
        folder_name = f"41mini_{variant}_{repeat:02d}"
        print(f"\n{folder_name}\n")
        folder = results_folder / folder_name
        folder.mkdir(parents=True, exist_ok=True)
        shutil.copy(prompt, folder / "01_01_user.md")
        cmd  = [sys.executable, "generated_adaptations/generator.py", f"--folder={str(folder)}"]  #, "--retries_test=1", "--retries_simulation=0"]
        result = subprocess.run(cmd, capture_output=True, text=True)  # even capture_output=False does not stream in real-time, so we rather capture it and print it later
        print(result.stdout)
        print(result.stderr)
        # generator.main([f"--folder={str(folder)}"])  # this also does not show the real-time output


41mini_default_02

Loaded 2 messages from generated_adaptations\farm\41mini_default_02.
Querying LLM with prompt 01_01_user.
LLM response saved to '01_02_llm.md'.
Code block saved to 'generated_adaptations\farm\41mini_default_02\code_01_02.py'.
TOKENS USED:
Input: 932, Output: 916 (reasoning: 0)
Response time (seconds): 15.3

Running tests: pytest generated_adaptations/tests -q --tb=short -rA --show-capture=no --color=no --example=farm --adaptation_name=41mini_default_02/code_01_02
Test exit code: 0
Running simulation for 'generated_adaptations\farm\41mini_default_02\code_01_02.py'.
  Run #1/2: python main.py farm/configs/default.yaml generated_adaptations/configs/generated.yaml farm/configs/config_no_battery.yaml DSL/drones.yaml --extra_config={"name": "41mini_default_02/code_01_02", "log_dir.append": "/41mini_default_02/code_01_02", "adaptation_name": "generated_adaptations.farm.41mini_default_02.code_01_02.SmartFarmAdaptation"} -s 1 -e 1
    Damage: 43

  Run #2/2: python main.py f

## Results

In [27]:
folders = list(results_folder.glob("41mini_*"))
print([f.stem for f in folders])

['41mini_default_01', '41mini_default_02', '41mini_default_03', '41mini_high_01', '41mini_high_02', '41mini_high_03', '41mini_less_01', '41mini_less_02', '41mini_less_03', '41mini_state_01', '41mini_state_02', '41mini_state_03']


In [28]:
summary = pd.DataFrame(columns=["llm", "params"])

In [29]:
for folder in folders:
    llm, params = folder.stem.split("_", 1)
    summary.loc[len(summary), ["llm", "params"]] = [llm, params]
    for file in (folder / "results").glob("*.txt"):
        name = file.stem.removeprefix("code_")
        if "_test_fail" in name:
            code = name.removesuffix("_test_fail")
            summary.loc[len(summary) - 1, code + "_test"] = "fail"
        elif "_test_pass" in name:
            code = name.removesuffix("_test_pass")
            summary.loc[len(summary) - 1, code + "_test"] = "pass"
        elif "_simulation_result" in name:
            code = name.removesuffix("_simulation_result")
            code = code.split("_")[0]  # only take the first part of the codefile name
            with open(file, "r") as f:
                result = f.read().strip()
                result = result.split(": ")[-1]  # TODO this is specific for the farm example
            summary.loc[len(summary) - 1, code + "_result"] = result
        else:
            print(f"Unknown file: {file}")
summary.fillna("", inplace=True)

In [30]:
summary.set_index(["llm", "params"], inplace=True)

In [31]:
summary = summary.reindex(sorted(summary.columns), axis=1)

In [32]:
summary

01_02_test 01_04_test 01_06_test 01_08_test 01_result  \
llm    params                                                             
41mini default_01       fail       fail       pass                123.5   
       default_02       pass                                       50.0   
       default_03       fail       pass                           123.5   
       high_01          fail       fail       pass                 52.0   
       high_02          fail       fail       pass                 52.5   
       high_03          fail       fail       fail       pass      52.5   
       less_01          pass                                      123.5   
       less_02          fail       pass                            88.0   
       less_03          pass                                       50.0   
       state_01         fail       fail       fail       pass     123.5   
       state_02         fail       pass                            56.5   
       state_03         fail       fail       fail       fail             

                  02_02_test 02_04_test 02_06_test 02_08_test 02_result  \
llm    params                                                             
41mini default_01       pass                                      123.5   
       default_02       fail       fail       fail       fail             
       default_03       fail       pass                           169.0   
       high_01          fail       fail       pass                 53.5   
       high_02          fail       pass                            52.5   
       high_03          fail       fail       pass                 52.5   
       less_01          fail       fail       fail       fail             
       less_02          fail       pass                            88.0   
       less_03          fail       pass                            46.5   
       state_01         fail       pass                            50.0   
       state_02         pass                                      268.5   
       state_03                                                           

                  03_02_test 03_04_test 03_06_test 03_result  
llm    params                                                 
41mini default_01       fail       fail       fail            
       default_02                                             
       default_03       fail       pass                162.5  
       high_01          pass                            53.5  
       high_02          fail       pass                 52.5  
       high_03          fail       pass                 69.5  
       less_01                                                
       less_02          pass                            81.5  
       less_03          pass                            37.5  
       state_01         pass                            53.5  
       state_02         pass                            56.5  
       state_03